In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import load_wine

from optimization_utils import (
    build_optuna_figures,
    compute_class_aucpr_scores,
    fit_model_pipeline,
    get_available_models,
    get_top_trials_dataframe,
    optimize_model,
)
from preprocessing_utils import prepare_dataset

In [ ]:
# Load the wine dataset
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Adding dummy categorical columns for robustness check
X["binary_cat"] = np.random.choice(["Yes", "No"], size=len(X))
X["multi_cat"] = np.random.choice(["Low", "Medium", "High"], size=len(X))

In [ ]:
prepared_data = prepare_dataset(X, y, test_size=0.2, random_state=42, stratify=True)

X_train = prepared_data.X_train
X_test = prepared_data.X_test
y_train = prepared_data.y_train
y_test = prepared_data.y_test
preprocessor = prepared_data.preprocessor

In [ ]:
model_name = "decision_tree"
available_models = get_available_models()

optimization_result = optimize_model(
    model_name=model_name,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train,
    n_trials=50,
    cv=5,
    random_state=42,
)

study = optimization_result.study
model_pipeline = optimization_result.best_pipeline
best_estimator = optimization_result.best_estimator
contour_params = optimization_result.contour_params

In [ ]:
figures = build_optuna_figures(study, contour_params=contour_params)

figures["optimization_history"].show()
figures["param_importances"].show()
figures["parallel_coordinate"].show()
figures["slice"].show()

In [ ]:
figures["edf"].show()

In [ ]:
if "contour" in figures:
    figures["contour"].show()

In [ ]:
figures["rank"].show()

In [ ]:
df_results = study.trials_dataframe()
top_trials = get_top_trials_dataframe(study, top_n=10)
display(top_trials)